# 04 — Grow-Shrink From Scratch

**Companion to `docs/05_discovery_algorithms.md` (Phase 4).**

We implement the Grow-Shrink (GS) algorithm and IAMB **from scratch**, narrating
each grow/shrink decision, and check the result against a network with a known
Markov Blanket. Seeing the phases run is the fastest way to understand *why* the
shrink phase exists.


In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2
rng = np.random.default_rng(1)
ALPHA = 0.05

## 0. Inline G-test and the test network (collider with a spouse)

In [2]:
def _table(x, y):
    xs, xi = np.unique(x, return_inverse=True)
    ys, yi = np.unique(y, return_inverse=True)
    t = np.zeros((len(xs), len(ys))); np.add.at(t, (xi, yi), 1); return t

def g_test(x, y, z=None):
    z = z or []; x = np.asarray(x); y = np.asarray(y)
    if not z:
        tabs = [_table(x, y)]
    else:
        zmat = np.column_stack([np.asarray(c).reshape(-1) for c in z])
        _, inv = np.unique(zmat, axis=0, return_inverse=True)
        tabs = [_table(x[inv==s], y[inv==s]) for s in range(inv.max()+1)]
    G, dof = 0.0, 0
    for tab in tabs:
        n = tab.sum()
        if n == 0: continue
        row = tab.sum(1, keepdims=True); col = tab.sum(0, keepdims=True)
        e = row @ col / n; nz = (tab > 0) & (e > 0)
        G += 2.0 * np.sum(tab[nz] * np.log(tab[nz] / e[nz]))
        dof += max(int(np.count_nonzero(row))-1,0) * max(int(np.count_nonzero(col))-1,0)
    return float(G), max(dof,1), float(chi2.sf(G, max(dof,1)))

def make_network(n=10000, seed=1):
    r = np.random.default_rng(seed)
    A = r.integers(0,2,n); B = r.integers(0,2,n); S = r.integers(0,2,n); D = r.integers(0,2,n)
    T = (r.random(n) < np.where(A==1,0.8,0.2)).astype(int)
    logit = -1.5 + 1.6*T + 1.6*B + 1.6*S
    C = (r.random(n) < 1/(1+np.exp(-logit))).astype(int)
    return pd.DataFrame({"A":A,"B":B,"S":S,"T":T,"C":C,"D":D}), {"A","C","B","S"}

df, true_mb = make_network()
cols = {k: df[k].to_numpy() for k in df.columns}

def stat(x, t, S):
    return g_test(cols[x], cols[t], [cols[v] for v in S])[0]
def is_indep(x, t, S):
    return g_test(cols[x], cols[t], [cols[v] for v in S])[2] > ALPHA

target = "T"
variables = [v for v in df.columns if v != target]
print("Variables:", variables, "  True MB(T):", sorted(true_mb))

Variables: ['A', 'B', 'S', 'C', 'D']   True MB(T): ['A', 'B', 'C', 'S']


## 1. Grow phase, narrated

Start empty. Repeatedly add the variable most associated with `T` that is still
**dependent** on `T` given the current set. Stop when none remain dependent.


In [3]:
S = []
print("GROW")
changed = True
while changed:
    changed = False
    cands = sorted([v for v in variables if v not in S], key=lambda x: stat(x, target, S), reverse=True)
    for x in cands:
        G, dof, p = g_test(cols[x], cols[target], [cols[v] for v in S])
        if p <= ALPHA:                       # dependent -> add
            S.append(x)
            print(f"  + add {x:>2}  (G={G:7.2f}, p={p:.2e})  S={S}")
            changed = True
            break
print("after grow:", S)

GROW
  + add  A  (G=3783.24, p=0.00e+00)  S=['A']
  + add  C  (G= 573.75, p=2.58e-125)  S=['A', 'C']
  + add  B  (G=  59.65, p=3.44e-12)  S=['A', 'C', 'B']
  + add  S  (G=  85.79, p=3.31e-15)  S=['A', 'C', 'B', 'S']
after grow: ['A', 'C', 'B', 'S']


## 2. Shrink phase — remove false positives

A variable can look dependent while `S` is incomplete, then become redundant
once the blanket fills in. Drop any `X` independent of `T` given the *rest*.


In [4]:
print("SHRINK")
changed = True
while changed:
    changed = False
    for x in list(S):
        rest = [v for v in S if v != x]
        if is_indep(x, target, rest):
            S.remove(x)
            print(f"  - drop {x:>2}  (independent given {rest})")
            changed = True
            break
    else:
        print("  nothing to drop")
print("\nFinal blanket:", sorted(S))
print("True blanket: ", sorted(true_mb))
print("MATCH:", set(S) == true_mb)

SHRINK


  nothing to drop

Final blanket: ['A', 'B', 'C', 'S']
True blanket:  ['A', 'B', 'C', 'S']
MATCH: True


## 3. Package as functions: GS and IAMB

IAMB differs only in the grow step: it always adds the **maximum-association**
variable, keeping the blanket (and thus conditioning sets) small throughout.


In [5]:
def grow_shrink(target):
    S = []
    changed = True
    while changed:
        changed = False
        cands = sorted([v for v in variables if v != target and v not in S],
                       key=lambda x: stat(x, target, S), reverse=True)
        for x in cands:
            if not is_indep(x, target, S):
                S.append(x); changed = True; break
    changed = True
    while changed:
        changed = False
        for x in list(S):
            rest = [v for v in S if v != x]
            if is_indep(x, target, rest):
                S.remove(x); changed = True; break
    return S

def iamb(target):
    S = []
    while True:
        cands = [v for v in variables if v != target and v not in S]
        if not cands: break
        best = max(cands, key=lambda x: stat(x, target, S))
        if not is_indep(best, target, S):
            S.append(best)
        else:
            break
    changed = True
    while changed:
        changed = False
        for x in list(S):
            rest = [v for v in S if v != x]
            if is_indep(x, target, rest):
                S.remove(x); changed = True; break
    return S

print("grow_shrink:", sorted(grow_shrink("T")))
print("iamb:       ", sorted(iamb("T")))

grow_shrink: ['A', 'B', 'C', 'S']


iamb:        ['A', 'B', 'C', 'S']


## 4. Sensitivity to α

Too low → true members dropped (under-selection); too high → noise slips in
(over-selection). Sweep and watch the blanket change.


In [6]:
def grow_shrink_alpha(target, alpha):
    global ALPHA
    saved = ALPHA; ALPHA = alpha
    try:
        return set(grow_shrink(target))
    finally:
        ALPHA = saved

for a in [0.001, 0.01, 0.05, 0.2, 0.5]:
    res = grow_shrink_alpha("T", a)
    extra = res - true_mb; missing = true_mb - res
    note = "exact" if not (extra or missing) else " ".join(
        ([f"+{sorted(extra)}"] if extra else []) + ([f"-{sorted(missing)}"] if missing else []))
    print(f"alpha={a:<6}: {sorted(res)}   {note}")

alpha=0.001 : ['A', 'B', 'C', 'S']   exact


alpha=0.01  : ['A', 'B', 'C', 'S']   exact


alpha=0.05  : ['A', 'B', 'C', 'S']   exact


alpha=0.2   : ['A', 'B', 'C', 'S']   exact


alpha=0.5   : ['A', 'B', 'C', 'S']   exact


### Takeaways

- **Grow** adds dependent variables; **shrink** removes ones the completed
  blanket makes redundant. Both phases are necessary.
- IAMB keeps conditioning sets small by adding the most-associated variable
  first — more reliable on real data.
- α is the key knob: too small under-selects, too large over-selects.

**Exercises:** `exercises/05_discovery_algorithms.md` (includes implementing
Inter-IAMB and the HITON/MMMB spouse step).
